# Day I10 — 신호처리 I: 표본화, 미분과 잡음, cutoff를 residual로 고르기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I10.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I10.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 앞뒤로 거르면 cutoff가 내려간다

In [ ]:
import numpy as np
from scipy import signal

fs = 70.0                                   # Winter 자료와 비슷한 표본화 주파수
b, a = signal.butter(2, 6, fs=fs)           # 2차 Butterworth, cutoff 6 Hz
w, h1 = signal.freqz(b, a, worN=[6.0], fs=fs)
print(f"한 번 통과(lfilter)  : 6 Hz에서 이득 {abs(h1[0]):.3f}")
print(f"앞뒤 통과(filtfilt)  : 6 Hz에서 이득 {abs(h1[0])**2:.3f}  (제곱이 된다)")

# 앞뒤 통과에서 이득이 0.707이 되는 실제 주파수 찾기
f = np.linspace(0.1, 20, 2000)
_, h = signal.freqz(b, a, worN=f, fs=fs)
f3 = f[np.argmin(abs(abs(h) ** 2 - 2 ** -0.5))]
print(f"보정 없이 filtfilt 하면 -3 dB 지점은 {f3:.2f} Hz")

C = (2 ** 0.5 - 1) ** 0.25
f_design = fs / np.pi * np.arctan(np.tan(np.pi * 6 / fs) / C)
b2, a2 = signal.butter(2, f_design, fs=fs)
_, h2 = signal.freqz(b2, a2, worN=[6.0], fs=fs)
print(f"보정계수 C = {C:.3f}, 설계 cutoff = {f_design:.2f} Hz (단순 근사 6/C = {6/C:.2f} Hz)")
print(f"보정 후 filtfilt: 6 Hz에서 이득 {abs(h2[0])**2:.3f}")

### 예시 2 — 필터 없이 미분하면 가속도는 잡음이다

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(10)
fs = 70.0
t = np.arange(0, 3, 1 / fs)
# 가상 마커 수직 위치(cm): stride 1 Hz의 1·2·3번째 조화파의 합
A = [(1, 3.0), (2, 1.5), (3, 0.6)]
true_y = sum(amp * np.sin(2 * np.pi * f * t) for f, amp in A)
true_acc = sum(-amp * (2 * np.pi * f) ** 2 * np.sin(2 * np.pi * f * t) for f, amp in A)
y = true_y + rng.normal(0, 0.2, len(t))          # 측정 잡음 SD 0.2 cm

def acc_central(x, fs):
    a = np.full_like(x, np.nan)
    a[1:-1] = (x[2:] - 2 * x[1:-1] + x[:-2]) * fs ** 2   # 2차 중앙차분
    return a

C = (2 ** 0.5 - 1) ** 0.25
def lowpass(x, fc, fs):
    f_design = fs / np.pi * np.arctan(np.tan(np.pi * fc / fs) / C)
    b, a = signal.butter(2, f_design, fs=fs)
    return signal.filtfilt(b, a, x)

inner = slice(10, -10)                            # 양 끝(필터·차분 가장자리)은 빼고 비교
def rms(e): return np.sqrt(np.nanmean(e[inner] ** 2))
print(f"위치 오차 rms        : {rms(y - true_y):7.2f} cm")
print(f"참 가속도 rms        : {rms(true_acc):7.1f} cm/s²")
print(f"필터 없이 미분한 오차 : {rms(acc_central(y, fs) - true_acc):7.1f} cm/s²")
for fc in [4, 6, 10]:
    print(f"{fc:2d} Hz로 거른 뒤 오차 : {rms(acc_central(lowpass(y, fc, fs), fs) - true_acc):7.1f} cm/s²")

### 예시 3 — WinterTableA1의 residual analysis: 좌표마다 답이 다르다

In [ ]:
import numpy as np
from scipy import signal

d = np.loadtxt("data/bmc/WinterTableA1.txt", skiprows=2)   # 머리글 2줄 건너뛰기
t = d[:, 1]                                                # 2열 = 시간(s)
fs = (len(t) - 1) / (t[-1] - t[0])
names = ["RIB", "HIP", "KNEE", "FIBULA", "ANKLE", "HEEL", "MT5", "TOE"]
print(f"frame {len(t)}개, fs = {fs:.2f} Hz")

C = (2 ** 0.5 - 1) ** 0.25                 # 앞뒤 두 번 통과 보정계수 0.802
def lowpass(x, fc):
    f_design = fs / np.pi * np.arctan(np.tan(np.pi * fc / fs) / C)
    b, a = signal.butter(2, f_design, fs=fs)
    return signal.filtfilt(b, a, x)

fcs = np.arange(0.5, 28.01, 0.1)           # cutoff 후보(Hz)
def winter_cutoff(x, tail):
    R = np.array([np.sqrt(np.mean((x - lowpass(x, fc)) ** 2)) for fc in fcs])  # RMS residual
    m = (fcs >= tail[0]) & (fcs <= tail[1])
    slope, icpt = np.polyfit(fcs[m], R[m], 1)     # 꼬리 직선의 0 Hz 절편 = 잡음 rms 추정
    return fcs[np.argmax(R <= icpt)], icpt        # residual이 처음 절편 높이로 내려오는 cutoff

print("마커    축  cutoff(꼬리 10–20 Hz)  cutoff(꼬리 15–28 Hz)  잡음 rms(cm)")
allfc = []
for k, nm in enumerate(names):
    for j, ax in enumerate("xy"):               # x = 진행 방향, y = 수직
        x = d[:, 2 + 2 * k + j]
        fc1, noise = winter_cutoff(x, (10, 20))
        fc2, _ = winter_cutoff(x, (15, 28))
        allfc.append(fc1)
        print(f"{nm:7s} {ax}        {fc1:4.1f}                  {fc2:4.1f}              {noise:.2f}")
print(f"꼬리 10–20 Hz 기준 범위 {min(allfc):.1f}–{max(allfc):.1f} Hz, 중앙값 {np.median(allfc):.1f} Hz")

### 예시 4 — autocorrelation으로 stride time 찾기 (gaitpdb)

In [ ]:
import numpy as np
from scipy.signal import find_peaks

fs = 100                                          # gaitpdb 표본화 주파수(format.txt)
d = np.loadtxt("data/gaitpdb/GaCo01_01.txt")      # 머리글 없음, 탭 구분 19열
left = d[:, 17]                                   # 18번째 열 = 왼발 총 힘(N)
both = d[:, 17] + d[:, 18]                        # 18 + 19번째 열 = 양발 합
seg = slice(10 * fs, 70 * fs)                     # 10–70초 구간(걷기 시작 부분 제외)

def autocorr(x):
    x = x - x.mean()                              # 평균을 빼지 않으면 모든 지연에서 상관이 부풀려진다
    r = np.correlate(x, x, mode="full")[len(x) - 1:]
    return r / r[0]                               # 지연 0에서 1

for name, x in [("왼발", left[seg]), ("양발 합", both[seg])]:
    r = autocorr(x)
    pk, _ = find_peaks(r[: 3 * fs], height=0.2)   # 3초 안의 봉우리들
    print(f"{name:5s}: 봉우리 지연(s) {np.round(pk / fs, 2)}  상관 {np.round(r[pk], 2)}")
    if name == "왼발":
        stride_ac = pk[0] / fs

# 비교 1: 20 N 문턱으로 찾은 왼발 heel strike 간격(= stride time)의 중앙값
x = left[seg]
on = x > 20
hs = np.where(~on[:-1] & on[1:])[0] + 1
print(f"문턱 방법 stride time 중앙값: {np.median(np.diff(hs)) / fs:.2f} s (간격 {len(hs) - 1}개)")

# 비교 2: 같은 신호의 스펙트럼에서 가장 센 성분
X = np.abs(np.fft.rfft(x - x.mean()))
f = np.fft.rfftfreq(len(x), 1 / fs)
print(f"스펙트럼 최대 성분 {f[np.argmax(X)]:.2f} Hz ↔ 1 / (autocorrelation stride) = {1 / stride_ac:.2f} Hz")